# YOLO training notes

This notebook is the one I used for the YOLO experiments on the parasite egg dataset.

Main flow:
- copy the zip files from Drive
- convert COCO labels to YOLO format
- write the split files + YAML
- train the selected model
- plot the summary results

## 1) Setup
Mount Drive and install the packages used in this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics -q
!pip install seaborn matplotlib pandas -q

## 2) Paths and dataset prep
I keep the main paths here first so the rest of the notebook is easier to edit.

In [ ]:
# imports + main paths

import os
import shutil
import json
import random
import yaml
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou

DRIVE_TRAIN_ZIP = '/content/drive/Shareddrives/Workspace/NKla/dataset.zip'

DRIVE_TEST_ZIP = '/content/drive/Shareddrives/Workspace/NKla/test.zip'

EXP_LOG_DIR = '/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments'

LOCAL_ROOT = '/content/temp_dataset'

DATA_SIZES = [2200, 4400, 6600, 8800, 11000]

CLASS_NAMES = [
    'Ascaris lumbricoides', 'Capillaria philippinensis', 'Enterobius vermicularis',
    'Fasciolopsis buski', 'Hookworm egg', 'Hymenolepis diminuta',
    'Hymenolepis nana', 'Opisthorchis viverrine', 'Paragonimus spp',
    'Taenia spp. egg', 'Trichuris trichiura'
]

os.makedirs(EXP_LOG_DIR, exist_ok=True)
print(f"log folder: {EXP_LOG_DIR}")

In [ ]:
# copy the zip files to local storage
import os
import shutil

def prepare_local_copy():
    # same idea as the other notebooks: copy to local first so file access is less annoying.

    local_train_dir = os.path.join(LOCAL_ROOT, 'train')
    if os.path.exists(local_train_dir): shutil.rmtree(local_train_dir)
    os.makedirs(local_train_dir)

    print("copying train zip from Drive...")
    shutil.copy(DRIVE_TRAIN_ZIP, '/content/train_temp.zip')

    print("extracting train data...")
    shutil.unpack_archive('/content/train_temp.zip', local_train_dir)
    os.remove('/content/train_temp.zip')

    if os.path.exists(os.path.join(local_train_dir, 'data')):
        print("Renaming 'data' to 'images'...")
        os.rename(os.path.join(local_train_dir, 'data'), os.path.join(local_train_dir, 'images'))

    local_test_dir = os.path.join(LOCAL_ROOT, 'test')
    if os.path.exists(local_test_dir): shutil.rmtree(local_test_dir)
    os.makedirs(local_test_dir)

    print("copying test zip from Drive...")
    shutil.copy(DRIVE_TEST_ZIP, '/content/test_temp.zip')

    print("extracting test data...")
    shutil.unpack_archive('/content/test_temp.zip', local_test_dir)
    os.remove('/content/test_temp.zip')

    if os.path.exists(os.path.join(local_test_dir, 'data')):
        os.rename(os.path.join(local_test_dir, 'data'), os.path.join(local_test_dir, 'images'))

    print(f"Dataset is ready under: {LOCAL_ROOT}")

    if os.path.exists(os.path.join(local_train_dir, 'images')):
        print(f"   Train images: {len(os.listdir(os.path.join(local_train_dir, 'images')))}")

    return local_train_dir, local_test_dir

LOCAL_TRAIN_DIR, LOCAL_TEST_DIR = prepare_local_copy()

## 3) Label conversion and split
This part converts the COCO boxes to YOLO text files and writes the dataset YAML.

In [ ]:
# convert COCO labels and write the split files
def coco_to_yolo_txt(json_path, img_dir, label_dir, prefix=""):

    print(f"{prefix}converting COCO labels to YOLO txt...")

    if not os.path.exists(label_dir): os.makedirs(label_dir)

    # read the json file
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"JSON file not found at {json_path}")
        return []

    img_info = {img['id']: img for img in data['images']}
    img_to_anns = {}
    for ann in data['annotations']:
        img_to_anns.setdefault(ann['image_id'], []).append(ann)

    converted_files = []

    for img_id, img_data in img_info.items():
        filename = img_data['file_name']
        img_w, img_h = img_data['width'], img_data['height']

        full_img_path = os.path.join(img_dir, filename)
        if not os.path.exists(full_img_path):
            full_img_path = full_img_path.replace('.jpg', '.png')

        if os.path.exists(full_img_path):
            converted_files.append(full_img_path)

            txt_filename = os.path.splitext(filename)[0] + ".txt"
            txt_path = os.path.join(label_dir, txt_filename)

            yolo_lines = []
            if img_id in img_to_anns:
                for ann in img_to_anns[img_id]:
                    cat_id = ann['category_id']
                    x, y, w, h = ann['bbox']

                    xc, yc, wn, hn = (x+w/2)/img_w, (y+h/2)/img_h, w/img_w, h/img_h

                    xc = max(0, min(1, xc))
                    yc = max(0, min(1, yc))
                    wn = max(0, min(1, wn))
                    hn = max(0, min(1, hn))

                    yolo_lines.append(f"{cat_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")

            with open(txt_path, 'w') as f:
                f.write('\n'.join(yolo_lines))

    print(f"{prefix} Converted {len(converted_files)} files.")
    return converted_files

def prepare_datasets():

    test_img_dir = os.path.join(LOCAL_TEST_DIR, 'images')
    test_json_path = os.path.join(LOCAL_TEST_DIR, 'test_labels_200.json')
    test_label_dir = os.path.join(LOCAL_TEST_DIR, 'labels')

    test_paths = coco_to_yolo_txt(test_json_path, test_img_dir, test_label_dir, prefix="[TEST]")

    test_list_path = os.path.join(LOCAL_ROOT, 'test_external.txt')
    with open(test_list_path, 'w') as f: f.write('\n'.join(test_paths))

    train_img_dir = os.path.join(LOCAL_TRAIN_DIR, 'images')

    train_json_path = os.path.join(LOCAL_TRAIN_DIR, 'labels.json')

    if not os.path.exists(train_json_path):
         candidates = glob(os.path.join(LOCAL_TRAIN_DIR, '**', '*.json'), recursive=True)
         if candidates:
             train_json_path = candidates[0]
             print(f"Found JSON at: {train_json_path}")

    train_label_dir = os.path.join(LOCAL_TRAIN_DIR, 'labels')

    all_train_images = coco_to_yolo_txt(train_json_path, train_img_dir, train_label_dir, prefix="[TRAIN]")

    if len(all_train_images) == 0:
        print("ERROR: No training images found or converted!")
        return [], test_list_path

    random.seed(42)
    random.shuffle(all_train_images)

    experiment_configs = []

    for size in DATA_SIZES:
        if size > len(all_train_images): continue

        current_subset = all_train_images[:size]
        train_imgs, val_imgs = train_test_split(current_subset, test_size=0.1, random_state=42)

        train_txt = os.path.join(LOCAL_ROOT, f'train_{size}.txt')
        val_txt = os.path.join(LOCAL_ROOT, f'val_{size}.txt')
        with open(train_txt, 'w') as f: f.write('\n'.join(train_imgs))
        with open(val_txt, 'w') as f: f.write('\n'.join(val_imgs))

        yaml_content = {
            'path': LOCAL_ROOT,
            'train': train_txt,
            'val': val_txt,
            'test': test_list_path,
            'nc': len(CLASS_NAMES),
            'names': {i: n for i, n in enumerate(CLASS_NAMES)}
        }
        yaml_path = os.path.join(LOCAL_ROOT, f'data_{size}.yaml')
        with open(yaml_path, 'w') as f: yaml.dump(yaml_content, f, sort_keys=False)

        print(f"Prepared Size {size}: Train={len(train_imgs)}, Val={len(val_imgs)}")

        experiment_configs.append({
            'size': size,
            'yaml': yaml_path,
            'train_count': len(train_imgs)
        })

    return experiment_configs, test_list_path

EXP_CONFIGS, TEST_LIST_PATH = prepare_datasets()

## 4) Training and summary
The last step is to train the selected YOLO model and plot the final trends.

In [ ]:
# run training for the selected model
import pandas as pd
from ultralytics import YOLO

TARGET_SIZE = 11000

selected_models = ['yolo11s.pt']

In [ ]:
def run_selected_training():
    print(f"dataset size: {TARGET_SIZE}")

    yaml_path = os.path.join(LOCAL_ROOT, f'data_{TARGET_SIZE}.yaml')

    if not os.path.exists(yaml_path):
        print(f"Config file not found: {yaml_path}")
        print("dataset yaml is missing")
        return

    results_log = []

    for model_name in selected_models:
        print(f"\n{'='*60}")
        print(f"training {model_name} with {TARGET_SIZE} images")
        print(f"{'='*60}")

        exp_name = f"{model_name.replace('.pt','')}_{TARGET_SIZE}"

        try: model = YOLO(model_name)
        except:
            print(f"Load failed: {model_name}")
            continue

        model.train(
            data=yaml_path,
            epochs=50,
            imgsz=640,
            batch=64,
            project=os.path.join(EXP_LOG_DIR, 'runs'),
            name=exp_name,
            exist_ok=True,
            verbose=False
        )

        print("running validation on the test split...")
        metrics = model.val(data=yaml_path, split='test', verbose=False)

        print(f"Validation mAP50-95 = {metrics.box.map:.4f}")

        results_log.append({
            'Model': model_name.replace('.pt','').upper(),
            'Dataset Size': TARGET_SIZE,
            'mAP50': metrics.box.map50,
            'mAP50-95': metrics.box.map,
            'Precision': metrics.box.mp,
            'Recall': metrics.box.mr
        })

    csv_name = f'results_size_{TARGET_SIZE}.csv'
    df = pd.DataFrame(results_log)
    df.to_csv(os.path.join(EXP_LOG_DIR, csv_name), index=False)
    print(f"\nSaved results to: {csv_name}")

run_selected_training()

In [ ]:
# plot the main graphs
csv_path = os.path.join(EXP_LOG_DIR, 'final_results_progress.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    sns.set_style("whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(24, 6))

    sns.lineplot(data=df, x='Dataset Size', y='mAP50-95', hue='Model', style='Model', markers=True, linewidth=3, ax=axes[0])
    axes[0].set_title('Accuracy (mAP@50-95)', fontsize=14)

    sns.lineplot(data=df, x='Dataset Size', y='mIoU', hue='Model', style='Model', markers=True, linewidth=3, ax=axes[1])
    axes[1].set_title('Localization (mIoU)', fontsize=14)

    sns.lineplot(data=df, x='Dataset Size', y='Recall', hue='Model', style='Model', markers=True, linewidth=3, ax=axes[2])
    axes[2].set_title('Sensitivity (Recall)', fontsize=14)

    plt.tight_layout()
    plt.savefig(os.path.join(EXP_LOG_DIR, 'final_benchmark_charts.png'))
    plt.show()

    print(f"Graphs saved to: {EXP_LOG_DIR}")
else:
    print("No results CSV found.")